**PS1 — Device Failure Prediction (3-day Binary) — TVM + GATE**

---
SageMaker Studio | MLflow | Feature Store | Model Registry | Model Monitor

Architecture : CUBIC MARS AWS v6
  Gold S3 (Delta) → Feature Store (offline+online) → Train 5 models + stacking
  → MLflow Experiment → MLflow Model Registry → SageMaker Endpoint
  → SageMaker Model Monitor (data quality + model quality)

Target  : will_fail_3d  (0=no failure within 3 days, 1=failure within 3 days)
Scope   : TVM + GATE only (VALIDATOR excluded: zero ServiceNow incidents in incident_root_cause)
Grain   : one row per DEVICE_ID per day
Run in  : SageMaker Studio JupyterLab  (kernel: Python 3 — Data Science 3.0)

---

In [ ]:
# ── CELL 1 : Install dependencies ─────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "s3fs", "pyarrow",
    "scikit-learn>=1.4",
    "xgboost>=2.0", "lightgbm>=4.3", "catboost>=1.2",
    "optuna>=3.6", "shap>=0.45",
    "mlflow>=2.13", "sagemaker-mlflow",
    "boto3", "sagemaker>=2.220,<3.0",
    "pandas>=2.0", "matplotlib", "seaborn",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
print("All packages installed.")

In [ ]:
# ── CELL 2 : Imports ──────────────────────────────────────────────────────────
import os, json, time, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap, optuna, boto3, sagemaker, joblib, tarfile

# deltalake removed: tables exported as Parquet (DeltaProtocolError on deletionVectors)
from sklearn.model_selection         import TimeSeriesSplit
from sklearn.preprocessing           import StandardScaler, LabelEncoder
from sklearn.pipeline                import Pipeline
from sklearn.linear_model            import LogisticRegression
from sklearn.ensemble                import (RandomForestClassifier,
                                             StackingClassifier)
from sklearn.calibration             import CalibratedClassifierCV
from sklearn.metrics                 import (roc_auc_score, roc_curve,
                                             f1_score, precision_score, recall_score,
                                             average_precision_score, precision_recall_curve,
                                             confusion_matrix, classification_report)
from xgboost   import XGBClassifier
from lightgbm  import LGBMClassifier, early_stopping, log_evaluation
from catboost  import CatBoostClassifier

import mlflow, mlflow.sklearn, mlflow.xgboost
from mlflow.models.signature         import infer_signature

try:
    from sagemaker import Session
except ImportError:
    from sagemaker.session import Session
try:
    from sagemaker import get_execution_role
except ImportError:
    from sagemaker.session import get_execution_role
from sagemaker.feature_store.feature_group      import FeatureGroup
from sagemaker.feature_store.feature_definition import (FeatureDefinition,
                                                        FeatureTypeEnum)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 60)
print("Imports OK.")

In [ ]:
# ── CELL 3 : Configuration ────────────────────────────────────────────────────
# Fill in your artifact bucket name (the dedicated artefacts bucket)
ARTIFACT_BUCKET = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"

GOLD_BUCKET = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX = "chicago/gold/device_ps1_daily"
GOLD_S3     = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION  = Session()
ROLE        = get_execution_role()
REGION      = boto3.Session().region_name
SM_CLIENT   = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT   = boto3.client("s3",        region_name=REGION)

# ── MLflow — SageMaker Managed Tracking Server
# Create once via AWS console:
#   SageMaker → MLflow → Create tracking server
#   Name: cubic-mars-mlflow-server-dev
#   Artifact store: s3://ARTIFACT_BUCKET/mlflow
#   Size: Small
# Then get the URL:
MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN

try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found — using local file fallback (EFS)")

mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)  # sagemaker-mlflow resolves arn: scheme
# Per-category names (experiment / model / registry / endpoint) are built inside
# CELL 6b and overridden at the top of each CATEGORY loop in CELL 10.
# Removed single-category stubs to avoid confusion with the Option B design.

# Gold schema version — bump when the feature set or label definition changes.
GOLD_VERSION = 6

print(f"Region      : {REGION}")
print(f"Gold S3     : {GOLD_S3}")
print(f"Artifacts   : s3://{ARTIFACT_BUCKET}/")
print(f"MLflow URI  : {MLFLOW_TRACKING_URI}")


In [ ]:
# ── CELL 4 : Load Gold table (Parquet on S3) ──────────────────────────────────
# G01 source: mars_dev.gold.device_ps1_daily exported via notebooks/export_gold_to_s3.py
# Column note: gold SQL writes "transit_day"; aliased to "event_date" below.

print(f"Loading Gold table from: {GOLD_S3}")

df = pd.read_parquet(GOLD_S3, storage_options={"client_kwargs": {"region_name": REGION}})
GOLD_VERSION = None   # Parquet has no version; use None for MLflow tags
DATA_SOURCE  = "parquet"

# gold SQL writes transit_day; alias to event_date used throughout notebook
if "transit_day" in df.columns and "event_date" not in df.columns:
    df = df.rename(columns={"transit_day": "event_date"})
df["event_date"] = pd.to_datetime(df["event_date"])

print(f"Parquet source  : confirmed")
print(f"Shape           : {df.shape}")
print(f"Date range      : {df['event_date'].min().date()}  →  {df['event_date'].max().date()}")
print(f"\nTarget distribution:\n{df['will_fail_3d'].value_counts()}")
print(f"Failure rate    : {df['will_fail_3d'].mean():.2%}")
print(f"\nDevice mix:\n{df.drop_duplicates('DEVICE_ID')['mars_device_category'].value_counts()}")

In [ ]:
# ── CELL 5 : EDA ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df["will_fail_3d"].value_counts().plot(
    kind="bar", ax=axes[0], color=["#43A047","#E53935"], edgecolor="white")
axes[0].set_title("Target: will_fail_3d")

(1 - df.isnull().mean()).sort_values().head(20).plot(
    kind="barh", ax=axes[1], color="#1565C0")
axes[1].set_title("Feature completeness (lowest 20)")

df.groupby("event_date")["will_fail_3d"].sum().plot(
    ax=axes[2], color="#E53935", linewidth=0.8)
axes[2].set_title("Daily failure count over time")

plt.tight_layout()
plt.savefig("/tmp/ps1_eda.png", dpi=100)
plt.show()
print("EDA saved → /tmp/ps1_eda.png")

In [ ]:
# ── CELL 6 : Feature engineering (global — all categories) ──────────────────
TARGET   = "will_fail_3d"

# R3-2: Exclude VALIDATOR — no ServiceNow incidents in incident_root_cause → permanent 0 label
df = df[df["mars_device_category"].isin(["TVM", "GATE"])].copy()
print(f"After VALIDATOR exclusion: {df.shape}  ({df['mars_device_category'].value_counts().to_dict()})")

# Fix 1 — Sort ONCE before any time-windowed features so rolling windows respect temporal order
df = df.sort_values(["DEVICE_ID", "event_date"]).reset_index(drop=True)

ID_COLS  = ["DEVICE_ID", "event_date", "split"]

# Derived rolling features — df already sorted by DEVICE_ID + event_date
for col, window in [("oos_flag", 7), ("oos_flag", 30), ("daily_event_count", 7)]:
    if col in df.columns:
        df[f"{col}_roll{window}"] = (
            df.groupby("DEVICE_ID")[col]
              .transform(lambda x: x.rolling(window, min_periods=1).sum()))

if "daily_event_count" in df.columns:
    df["event_rate_zscore"] = (
        df.groupby("DEVICE_ID")["daily_event_count"]
          .transform(lambda x: (x - x.mean()) / (x.std() + 1e-6)))

if "use_revenue_decline_flag" in df.columns:
    df["use_revenue_decline_flag"] = df["use_revenue_decline_flag"].astype(int)

if "use_revenue_cents" in df.columns:
    # df already sorted globally — no redundant per-column sort needed
    df["use_revenue_cents_roll7"] = (
        df.groupby("DEVICE_ID")["use_revenue_cents"]
          .transform(lambda x: x.rolling(7, min_periods=1).mean()))

print(f"use_revenue columns: {[c for c in df.columns if c.startswith('use_revenue') or c.startswith('use_txn')]}")

# Fix 2 — Explicit leak exclusion: columns derived from or coincident with the failure outcome
# Including any of these would let the model see the answer and fake high AUC/AP.
LABEL_LEAK_COLS = [
    "failure_level",     # directly encodes the outcome
    "is_chargeable",     # same-day OOS verdict — not known at prediction time
    "is_hardware_oos",   # same-day OOS flag
    "is_hardware_event", # same-day hardware event
    "duration_min",      # OOS duration — known only after the event resolves
    "oos_duration_min",  # alias
    "will_fail_7d",      # old label — must never appear as a feature
]

FEATURE_COLS = [
    c for c in df.columns
    if c not in ID_COLS + [TARGET] + LABEL_LEAK_COLS
    and pd.api.types.is_numeric_dtype(df[c])
]
print(f"Features : {len(FEATURE_COLS)}")
print(f"Leak cols present in df (excluded): {[c for c in LABEL_LEAK_COLS if c in df.columns]}")

# Fix 3 — NaN fill is done per-category on TRAIN-ONLY inside CELL 10.
# Do NOT fill here — median computed on the full df leaks future (val/test) distribution into train.
print("Feature engineering complete — per-category splits in CELL 6b")


In [ ]:
# ── CELL 6b : Per-category temporal splits & resource config ─────────────────
# TVM and GATE trained as SEPARATE models.
# Each gets its own: split, scale_pos_weight, MLflow experiment,
# Feature Store group, Model Registry, and SageMaker endpoint.

CATEGORIES = {}
for cat in ["TVM", "GATE"]:
    cat_df = df[df["mars_device_category"] == cat].sort_values(["DEVICE_ID", "event_date"])

    tr = cat_df[cat_df["event_date"] <  "2025-10-01"]
    va = cat_df[(cat_df["event_date"] >= "2025-10-01") & (cat_df["event_date"] < "2026-01-01")]
    te = cat_df[cat_df["event_date"] >= "2026-01-01"]

    n0, n1 = (tr[TARGET] == 0).sum(), (tr[TARGET] == 1).sum()

    CATEGORIES[cat] = {
        "train_df": tr, "val_df": va, "test_df": te,
        "X_train": tr[FEATURE_COLS].values, "y_train": tr[TARGET].values,
        "X_val":   va[FEATURE_COLS].values, "y_val":   va[TARGET].values,
        "X_test":  te[FEATURE_COLS].values, "y_test":  te[TARGET].values,
        "X_cv":    np.vstack([tr[FEATURE_COLS].values, va[FEATURE_COLS].values]),
        "y_cv":    np.concatenate([tr[TARGET].values, va[TARGET].values]),
        "scale_pos_weight": round(n0 / max(n1, 1), 2),
        "neg": n0, "pos": n1,
        "experiment_name": f"chicago-ps1-3d-{cat.lower()}-failure",
        "model_name":      f"ps1-3d-{cat.lower()}-device-failure",
        "registry_name":   f"chicago-ps1-3d-{cat.lower()}-failure",
        "fs_group":        f"chicago-ps1-3d-{cat.lower()}-features",
        "endpoint":        f"chicago-ps1-3d-{cat.lower()}-failure-v1",
        # Recall floor for threshold tuning on val PR curve.
        # GATE: only 0.60% positive rate (scale_pos_weight ~166) —
        # recall=0.80 may not be achievable; use 0.70 as the floor.
        "recall_floor": 0.80 if cat == "TVM" else 0.70,
    }
    mlflow.set_experiment(CATEGORIES[cat]["experiment_name"])

    print(f"\n── {cat} ──────────────────────────────────────────────────────")
    print(f"  Train  : {len(tr):>8,}  ({tr['event_date'].min().date()} -> {tr['event_date'].max().date()})")
    print(f"  Val    : {len(va):>8,}")
    print(f"  Test   : {len(te):>8,}")
    print(f"  Pos rate (train): {n1/max(len(tr),1):.2%}  ({n1:,} positives)")
    print(f"  scale_pos_weight: {CATEGORIES[cat]['scale_pos_weight']}")
    print(f"  MLflow experiment: {CATEGORIES[cat]['experiment_name']}")


In [ ]:
# ── CELL 7 : SageMaker Feature Store — create Feature Groups (per category) ──
for CATEGORY, cat in CATEGORIES.items():
    tr = cat["train_df"]
    fs_df = tr[FEATURE_COLS + [TARGET, "DEVICE_ID", "event_date"]].copy()
    fs_df["event_time"] = fs_df["event_date"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    fs_df["DEVICE_ID"]  = fs_df["DEVICE_ID"].astype(str)
    fs_df               = fs_df.drop(columns=["event_date"])

    def dtype_to_fs_type(dtype):
        if pd.api.types.is_float_dtype(dtype):   return FeatureTypeEnum.FRACTIONAL
        if pd.api.types.is_integer_dtype(dtype): return FeatureTypeEnum.INTEGRAL
        return FeatureTypeEnum.STRING

    feature_definitions = [
        FeatureDefinition(feature_name=col, feature_type=dtype_to_fs_type(fs_df[col].dtype))
        for col in fs_df.columns
    ]
    fg = FeatureGroup(name=cat["fs_group"], sagemaker_session=SM_SESSION,
                      feature_definitions=feature_definitions)
    try:
        fg.create(
            s3_uri                  = f"s3://{ARTIFACT_BUCKET}/feature-store/{cat['fs_group']}",
            record_identifier_name  = "DEVICE_ID",
            event_time_feature_name = "event_time",
            role_arn                = ROLE,
            enable_online_store     = True,
            description             = f"PS1-3d {CATEGORY} daily device features — Chicago Ventra v6",
        )
        status = ""
        while status != "Created":
            time.sleep(15)
            status = fg.describe().get("FeatureGroupStatus")
            print(f"  [{CATEGORY}] Feature Group status: {status}")
        print(f"  [{CATEGORY}] Feature Group created: {cat['fs_group']}")
    except Exception as e:
        print(f"  [{CATEGORY}] Feature Group already exists or error: {e}")
    cat["feature_group"] = fg
    cat["fs_df"] = fs_df


In [ ]:
# ── CELL 8 : Ingest training features into Feature Store (per category) ──────
for CATEGORY, cat in CATEGORIES.items():
    print(f"[{CATEGORY}] Ingesting {len(cat['fs_df']):,} records...")
    cat["feature_group"].ingest(
        data_frame=cat["fs_df"].reset_index(drop=True), max_workers=4, wait=True)
    print(f"  [{CATEGORY}] Done -> {cat['fs_group']}")
    sample_id = cat["fs_df"]["DEVICE_ID"].iloc[0]
    rec = cat["feature_group"].get_record(record_identifier_value_as_string=sample_id)
    print(f"  [{CATEGORY}] Online store sample ({sample_id}):")
    for fv in rec[:3]:
        print(f"    {fv['FeatureName']}: {fv['ValueAsString']}")


In [ ]:
# ── CELL 9 : Helper functions (category-aware via loop globals) ──────────────
# X_train/val/test, y_train/val/test, SCALE_POS_WEIGHT, COMMON_TAGS
# are reassigned at the start of each CATEGORY iteration.

def make_common_tags(category):
    return {
        "ps"           : "PS1_3d",
        "category"     : category,
        "target"       : "will_fail_3d",
        "city"         : "chicago",
        "gold_version" : str(GOLD_VERSION),
        "grain"        : "device_day",
        "architecture" : "CUBIC-MARS-AWS-v6",
    }

def evaluate(model, X, y, split="test"):
    prob = model.predict_proba(X)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {
        f"{split}_auc"  : round(roc_auc_score(y, prob), 4),
        f"{split}_ap"   : round(average_precision_score(y, prob), 4),
        f"{split}_f1"   : round(f1_score(y, pred), 4),
        f"{split}_prec" : round(precision_score(y, pred, zero_division=0), 4),
        f"{split}_rec"  : round(recall_score(y, pred, zero_division=0), 4),
    }

def log_cm(model, X, y, name, split="test"):
    pred = (model.predict_proba(X)[:, 1] >= 0.5).astype(int)
    cm   = confusion_matrix(y, pred)
    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Fail","Fail"], yticklabels=["No Fail","Fail"])
    ax.set_title(f"{name} — Confusion Matrix ({split})")
    path = f"/tmp/cm_{name}_{split}.png"
    fig.savefig(path, bbox_inches="tight", dpi=80); plt.show(); plt.close(fig)
    mlflow.log_artifact(path)

def plot_roc_pr(model, name):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    colors = {"train": "#1565C0", "val": "#E65100", "test": "#2E7D32"}
    for label, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
        prob        = model.predict_proba(X)[:, 1]
        fpr, tpr, _ = roc_curve(y, prob)
        pr, rc, _   = precision_recall_curve(y, prob)
        col         = colors[label]
        axes[0].plot(fpr, tpr, color=col, lw=1.6, label=f"{label}  AUC={roc_auc_score(y,prob):.4f}")
        axes[1].plot(rc,  pr,  color=col, lw=1.6, label=f"{label}  AP={average_precision_score(y,prob):.4f}")
    axes[0].plot([0,1],[0,1],"k--", lw=0.7)
    axes[0].set(xlabel="FPR", ylabel="TPR", title=f"{name} — ROC"); axes[0].legend(fontsize=9)
    axes[1].set(xlabel="Recall", ylabel="Precision", title=f"{name} — PR"); axes[1].legend(fontsize=9)
    fig.suptitle(f"{name}  |  Train/Val/Test", fontsize=11, y=1.01); fig.tight_layout()
    path = f"/tmp/roc_pr_{name}.png"
    fig.savefig(path, dpi=100, bbox_inches="tight"); plt.show(); plt.close(fig)
    mlflow.log_artifact(path)

def log_feat_importance(importances, name, top_n=20):
    fi = pd.Series(importances, index=FEATURE_COLS).nlargest(top_n)
    fig, ax = plt.subplots(figsize=(9, 5))
    fi.sort_values().plot(kind="barh", ax=ax, color="#1565C0", edgecolor="white")
    ax.set_title(f"{name} — Top {top_n} Feature Importances"); ax.set_xlabel("Importance")
    fig.tight_layout()
    path = f"/tmp/fi_{name}.png"
    fig.savefig(path, dpi=100); plt.show(); plt.close(fig); mlflow.log_artifact(path)
    print(f"\n-- {name} -- Top 10"); print(fi.head(10).to_string())

def log_clf_report(model, name):
    for split, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
        pred = (model.predict_proba(X)[:, 1] >= 0.5).astype(int)
        print(f"\n-- {name} [{split}]")
        print(classification_report(y, pred, target_names=["No Fail", "Fail"], digits=4))
    pred_test = (model.predict_proba(X_test)[:, 1] >= 0.5).astype(int)
    rep = classification_report(y_test, pred_test, target_names=["No Fail","Fail"], digits=4)
    path = f"/tmp/report_{name}.txt"
    with open(path, "w") as fh: fh.write(rep)
    mlflow.log_artifact(path)

print("Helpers defined.")


In [ ]:
# ── CELL 10 : Per-category training loop (TVM then GATE) ────────────────────
# Trains 5 base models + Optuna XGBoost + Stacking for EACH category separately.
# Helpers use X_train/y_train etc. as module-level names — reassigned each iteration.

CATEGORY_RESULTS = {}

for CATEGORY in ["TVM", "GATE"]:
    cat = CATEGORIES[CATEGORY]
    X_train = cat["X_train"]; y_train = cat["y_train"]
    X_val   = cat["X_val"];   y_val   = cat["y_val"]
    X_test  = cat["X_test"];  y_test  = cat["y_test"]
    X_cv    = cat["X_cv"];    y_cv    = cat["y_cv"]
    train_df = cat["train_df"]; val_df = cat["val_df"]; test_df = cat["test_df"]

    # Fix 3 — Imputation: fit median on TRAIN only, apply to all splits
    # Avoids leaking val/test distribution into training imputation.
    train_median = np.nanmedian(X_train, axis=0)
    X_train = np.where(np.isnan(X_train), train_median, X_train)
    X_val   = np.where(np.isnan(X_val),   train_median, X_val)
    X_test  = np.where(np.isnan(X_test),  train_median, X_test)
    X_cv    = np.where(np.isnan(X_cv),    train_median, X_cv)
    cat["X_train"] = X_train; cat["X_val"] = X_val
    cat["X_test"]  = X_test;  cat["X_cv"]  = X_cv
    SCALE_POS_WEIGHT  = cat["scale_pos_weight"]
    EXPERIMENT_NAME   = cat["experiment_name"]
    MLFLOW_MODEL_NAME = cat["model_name"]
    MODEL_REGISTRY_NAME = cat["registry_name"]
    FS_GROUP_NAME     = cat["fs_group"]
    ENDPOINT_NAME     = cat["endpoint"]
    # Fix 4 — run_date tag prevents stale run accumulation across reruns
    COMMON_TAGS       = {**make_common_tags(CATEGORY),
                         "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}
    mlflow.set_experiment(EXPERIMENT_NAME)
    tag = CATEGORY.lower()

    print(f"\n{'='*65}")
    print(f"  Training: {CATEGORY}")
    print(f"{'='*65}")
    print(f"  Rows : train={len(X_train):,}  val={len(X_val):,}  test={len(X_test):,}")
    print(f"  Pos rate (train): {y_train.mean():.2%}  spw={SCALE_POS_WEIGHT}")

    # ── MODEL 1: Logistic Regression ─────────────────────────────────────────
    with mlflow.start_run(run_name=f"lr_{tag}") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "linear_baseline"})
        lr = Pipeline([("sc", StandardScaler()),
                        ("clf", LogisticRegression(C=0.1, class_weight="balanced",
                                                   max_iter=1000, solver="lbfgs", random_state=42))])
        lr.fit(X_train, y_train)
        vm = evaluate(lr, X_val, y_val, "val"); tm = evaluate(lr, X_test, y_test, "test")
        mlflow.log_params({"C":0.1,"class_weight":"balanced","scaler":"standard"})
        mlflow.log_metrics({**vm, **tm})
        log_cm(lr, X_test, y_test, f"lr_{tag}"); plot_roc_pr(lr, f"lr_{tag}")
        log_feat_importance(np.abs(lr.named_steps["clf"].coef_[0]), f"lr_{tag}")
        log_clf_report(lr, f"lr_{tag}")
        mlflow.sklearn.log_model(lr, "model", signature=infer_signature(X_train, lr.predict_proba(X_train)),
                                 input_example=X_train[:3])
        LR_RUN = run.info.run_id
    print(f"  LR  AUC={tm['test_auc']}  AP={tm['test_ap']}  F1={tm['test_f1']}")

    # ── MODEL 2: Random Forest ────────────────────────────────────────────────
    with mlflow.start_run(run_name=f"rf_{tag}") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "bagging_ensemble"})
        rf = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=20,
                                     class_weight="balanced_subsample", n_jobs=-1, random_state=42)
        rf.fit(X_train, y_train)
        vm = evaluate(rf, X_val, y_val, "val"); tm = evaluate(rf, X_test, y_test, "test")
        mlflow.log_params({"n_estimators":300,"max_depth":12,"class_weight":"balanced_subsample"})
        mlflow.log_metrics({**vm, **tm})
        log_cm(rf, X_test, y_test, f"rf_{tag}"); plot_roc_pr(rf, f"rf_{tag}")
        log_feat_importance(rf.feature_importances_, f"rf_{tag}"); log_clf_report(rf, f"rf_{tag}")
        mlflow.sklearn.log_model(rf, "model", signature=infer_signature(X_train, rf.predict_proba(X_train)),
                                 input_example=X_train[:3])
        RF_RUN = run.info.run_id
    print(f"  RF  AUC={tm['test_auc']}  AP={tm['test_ap']}  F1={tm['test_f1']}")

    # ── MODEL 3: XGBoost ─────────────────────────────────────────────────────
    XGB_PARAMS = dict(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, min_child_weight=10, gamma=1.0, reg_alpha=0.1,
                      reg_lambda=1.0, scale_pos_weight=SCALE_POS_WEIGHT, eval_metric="auc",
                      early_stopping_rounds=40, tree_method="hist", random_state=42, n_jobs=-1)
    with mlflow.start_run(run_name=f"xgb_{tag}") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "gradient_boosting_xgb"})
        xgb = XGBClassifier(**XGB_PARAMS)
        xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
        vm = evaluate(xgb, X_val, y_val, "val"); tm = evaluate(xgb, X_test, y_test, "test")
        mlflow.log_params({**XGB_PARAMS, "best_iter": xgb.best_iteration})
        mlflow.log_metrics({**vm, **tm})
        log_cm(xgb, X_test, y_test, f"xgb_{tag}"); plot_roc_pr(xgb, f"xgb_{tag}")
        log_feat_importance(xgb.feature_importances_, f"xgb_{tag}"); log_clf_report(xgb, f"xgb_{tag}")
        sig = infer_signature(pd.DataFrame(X_train, columns=FEATURE_COLS), xgb.predict_proba(X_train))
        mlflow.xgboost.log_model(xgb, "model", signature=sig,
                                  input_example=pd.DataFrame(X_train[:3], columns=FEATURE_COLS))
        XGB_RUN = run.info.run_id
    print(f"  XGB AUC={tm['test_auc']}  AP={tm['test_ap']}  F1={tm['test_f1']}  best_iter={xgb.best_iteration}")

    # ── MODEL 4: LightGBM ─────────────────────────────────────────────────────
    LGB_PARAMS = dict(n_estimators=600, num_leaves=63, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
                      scale_pos_weight=SCALE_POS_WEIGHT, n_jobs=-1, random_state=42, verbose=-1)
    with mlflow.start_run(run_name=f"lgb_{tag}") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "gradient_boosting_lgbm"})
        lgb = LGBMClassifier(**LGB_PARAMS)
        lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                callbacks=[early_stopping(40, verbose=False), log_evaluation(100)])
        vm = evaluate(lgb, X_val, y_val, "val"); tm = evaluate(lgb, X_test, y_test, "test")
        mlflow.log_params({**LGB_PARAMS, "best_iter": lgb.best_iteration_})
        mlflow.log_metrics({**vm, **tm})
        log_cm(lgb, X_test, y_test, f"lgb_{tag}"); plot_roc_pr(lgb, f"lgb_{tag}")
        log_feat_importance(lgb.feature_importances_, f"lgb_{tag}"); log_clf_report(lgb, f"lgb_{tag}")
        mlflow.sklearn.log_model(lgb, "model", signature=infer_signature(X_train, lgb.predict_proba(X_train)),
                                 input_example=X_train[:3],
                                 serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
        LGB_RUN = run.info.run_id
    print(f"  LGB AUC={tm['test_auc']}  AP={tm['test_ap']}  F1={tm['test_f1']}  best_iter={lgb.best_iteration_}")

    # ── MODEL 5: CatBoost ─────────────────────────────────────────────────────
    cat_idx = [i for i, c in enumerate(FEATURE_COLS)
               if any(k in c for k in ("device_type","facility","subsystem","enc"))]
    CB_PARAMS = dict(iterations=600, depth=7, learning_rate=0.05, l2_leaf_reg=3,
                     scale_pos_weight=SCALE_POS_WEIGHT, eval_metric="AUC",
                     early_stopping_rounds=40, random_seed=42, task_type="CPU", verbose=100)
    with mlflow.start_run(run_name=f"cb_{tag}") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "gradient_boosting_catboost"})
        cb = CatBoostClassifier(**CB_PARAMS)
        cb.fit(X_train, y_train, eval_set=(X_val, y_val), cat_features=cat_idx if cat_idx else None)
        vm = evaluate(cb, X_val, y_val, "val"); tm = evaluate(cb, X_test, y_test, "test")
        mlflow.log_params({**{k:v for k,v in CB_PARAMS.items() if k!="verbose"},
                           "best_iter": cb.best_iteration_, "cat_features_count": len(cat_idx)})
        mlflow.log_metrics({**vm, **tm})
        log_cm(cb, X_test, y_test, f"cb_{tag}"); plot_roc_pr(cb, f"cb_{tag}")
        log_feat_importance(cb.get_feature_importance(), f"cb_{tag}"); log_clf_report(cb, f"cb_{tag}")
        cb.save_model(f"/tmp/ps1_{tag}_catboost.cbm")
        mlflow.log_artifact(f"/tmp/ps1_{tag}_catboost.cbm", "catboost_native")
        mlflow.sklearn.log_model(cb, "model", signature=infer_signature(X_train, cb.predict_proba(X_train)),
                                 input_example=X_train[:3],
                                 serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
        CB_RUN = run.info.run_id
    print(f"  CB  AUC={tm['test_auc']}  AP={tm['test_ap']}  F1={tm['test_f1']}  best_iter={cb.best_iteration_}")

    # ── Optuna XGBoost ────────────────────────────────────────────────────────
    TS_CV = TimeSeriesSplit(n_splits=5)
    def optuna_objective(trial):
        p = dict(n_estimators=trial.suggest_int("n_estimators",200,800),
                 max_depth=trial.suggest_int("max_depth",4,9),
                 learning_rate=trial.suggest_float("learning_rate",0.01,0.2,log=True),
                 subsample=trial.suggest_float("subsample",0.5,1.0),
                 colsample_bytree=trial.suggest_float("colsample_bytree",0.5,1.0),
                 min_child_weight=trial.suggest_int("min_child_weight",1,30),
                 gamma=trial.suggest_float("gamma",0.0,3.0),
                 reg_alpha=trial.suggest_float("reg_alpha",1e-4,10.,log=True),
                 reg_lambda=trial.suggest_float("reg_lambda",1e-4,10.,log=True),
                 scale_pos_weight=SCALE_POS_WEIGHT, tree_method="hist", random_state=42, n_jobs=-1)
        scores = []
        for step, (tr, va) in enumerate(TS_CV.split(X_cv)):
            m = XGBClassifier(**p)
            m.fit(X_cv[tr], y_cv[tr], eval_set=[(X_cv[va],y_cv[va])], verbose=False)
            scores.append(roc_auc_score(y_cv[va], m.predict_proba(X_cv[va])[:,1]))
            trial.report(np.mean(scores), step=step)
            if trial.should_prune(): raise optuna.TrialPruned()
        return np.mean(scores)

    study = optuna.create_study(direction="maximize", study_name=f"ps1-xgb-{tag}",
                                 pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1))
    study.optimize(optuna_objective, n_trials=50, show_progress_bar=True)
    BEST_PARAMS = {**study.best_params, "scale_pos_weight": SCALE_POS_WEIGHT,
                   "tree_method": "hist", "random_state": 42, "n_jobs": -1}

    with mlflow.start_run(run_name=f"xgb_optuna_{tag}") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "xgboost_optuna_tuned"})
        mlflow.log_params({**BEST_PARAMS, "optuna_trials":50, "optuna_cv_folds":5,
                           "optuna_best_cv": round(study.best_value,4), "smote": False})
        xgb_tuned = XGBClassifier(**BEST_PARAMS)
        xgb_tuned.fit(X_cv, y_cv, verbose=False)
        vm = evaluate(xgb_tuned, X_val, y_val, "val"); tm = evaluate(xgb_tuned, X_test, y_test, "test")
        mlflow.log_metrics({**vm, **tm, "optuna_cv_auc": round(study.best_value,4)})
        log_cm(xgb_tuned, X_test, y_test, f"xgb_optuna_{tag}"); plot_roc_pr(xgb_tuned, f"xgb_optuna_{tag}")
        log_feat_importance(xgb_tuned.feature_importances_, f"xgb_optuna_{tag}")
        log_clf_report(xgb_tuned, f"xgb_optuna_{tag}")
        for fname, fig_fn in [(f"optuna_history_{tag}.png", optuna.visualization.matplotlib.plot_optimization_history),
                               (f"optuna_param_import_{tag}.png", optuna.visualization.matplotlib.plot_param_importances)]:
            try:
                fig_fn(study); plt.tight_layout(); plt.savefig(f"/tmp/{fname}", dpi=80); plt.close()
                mlflow.log_artifact(f"/tmp/{fname}")
            except Exception: pass
        sig = infer_signature(pd.DataFrame(X_train, columns=FEATURE_COLS), xgb_tuned.predict_proba(X_train))
        mlflow.xgboost.log_model(xgb_tuned, "model", signature=sig,
                                  input_example=pd.DataFrame(X_train[:3], columns=FEATURE_COLS))
        XGB_TUNED_RUN = run.info.run_id
    print(f"  XGB_Optuna CV={study.best_value:.4f}  test AUC={tm['test_auc']}")

    # ── Stacking Ensemble ─────────────────────────────────────────────────────
    with mlflow.start_run(run_name=f"stack_{tag}") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "stacking_ensemble"})
        mlflow.log_params({"base_models":"LR+RF+XGB_tuned+LGB+CatBoost",
                           "final_estimator":"LogisticRegression+isotonic",
                           "cv_strategy":"TimeSeriesSplit(5)", "passthrough":True, "smote":False})
        base = [
            ("lr",  Pipeline([("sc",StandardScaler()),("clf",LogisticRegression(C=0.1,class_weight="balanced",max_iter=1000,random_state=42))])),
            ("rf",  RandomForestClassifier(n_estimators=200,max_depth=10,class_weight="balanced_subsample",n_jobs=-1,random_state=42)),
            ("xgb", XGBClassifier(**{**BEST_PARAMS,"n_estimators":300},verbosity=0)),
            ("lgb", LGBMClassifier(n_estimators=300,learning_rate=0.05,scale_pos_weight=SCALE_POS_WEIGHT,n_jobs=-1,random_state=42,verbose=-1)),
            ("cb",  CatBoostClassifier(iterations=300,learning_rate=0.05,scale_pos_weight=SCALE_POS_WEIGHT,random_seed=42,verbose=0)),
        ]
        stack = StackingClassifier(
            estimators=base,
            final_estimator=CalibratedClassifierCV(LogisticRegression(C=1.0,max_iter=500),cv=3,method="isotonic"),
            cv=TimeSeriesSplit(n_splits=5), stack_method="predict_proba", passthrough=True, n_jobs=-1)
        stack.fit(X_cv, y_cv)
        vm = evaluate(stack, X_val, y_val, "val"); tm = evaluate(stack, X_test, y_test, "test")
        mlflow.log_metrics({**vm, **tm})
        log_cm(stack, X_test, y_test, f"stack_{tag}"); plot_roc_pr(stack, f"stack_{tag}")
        log_clf_report(stack, f"stack_{tag}")
        mlflow.sklearn.log_model(stack, "model", signature=infer_signature(X_train, stack.predict_proba(X_train)),
                                 input_example=X_train[:3],
                                 serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE)
        STACK_RUN = run.info.run_id
    print(f"  Stack AUC={tm['test_auc']}  AP={tm['test_ap']}  F1={tm['test_f1']}")

    # ── SHAP ─────────────────────────────────────────────────────────────────
    print(f"  Computing SHAP ({CATEGORY})...")
    X_shap = pd.DataFrame(X_test[:2000], columns=FEATURE_COLS)
    explainer = shap.TreeExplainer(xgb_tuned); shap_values = explainer(X_shap)
    with mlflow.start_run(run_name=f"xgb_shap_{tag}") as run:
        mlflow.set_tags({**COMMON_TAGS, "model_type":"shap_analysis", "parent_run":XGB_TUNED_RUN})
        for sname, sfn in [("bar",shap.plots.bar),("beeswarm",shap.plots.beeswarm)]:
            sfn(shap_values, max_display=20, show=False)
            fig=plt.gcf(); fig.tight_layout()
            fpath=f"/tmp/shap_{sname}_{tag}.png"; fig.savefig(fpath,dpi=100); plt.close(fig)
            mlflow.log_artifact(fpath)
        top_feat = pd.Series(np.abs(shap_values.values).mean(0), index=FEATURE_COLS).idxmax()
        shap.plots.scatter(shap_values[:,top_feat], color=shap_values, show=False)
        fig=plt.gcf(); fig.tight_layout()
        fpath=f"/tmp/shap_dep_{tag}.png"; fig.savefig(fpath,dpi=100); plt.close(fig)
        mlflow.log_artifact(fpath)
        shap_imp = (pd.Series(np.abs(shap_values.values).mean(0), index=FEATURE_COLS)
                      .sort_values(ascending=False).reset_index()
                      .rename(columns={"index":"feature",0:"mean_abs_shap"}))
        shap_imp.to_csv(f"/tmp/shap_importance_{tag}.csv", index=False)
        mlflow.log_artifact(f"/tmp/shap_importance_{tag}.csv")
        mlflow.log_param("top_shap_feature", top_feat)
    print(f"  Top SHAP ({CATEGORY}): {top_feat}")

    # ── Stage 2 severity ─────────────────────────────────────────────────────
    if "failure_level" in test_df.columns:
        sev_tr = train_df[train_df["will_fail_3d"]==1].dropna(subset=["failure_level"])
        sev_te = test_df[test_df["will_fail_3d"]==1].dropna(subset=["failure_level"])
        # Guard: skip Stage 2 if any severity class has < 20 training samples.
        # GATE has failure_levels 2/3/4 with ~6/7/6 rows in source — multiclass is meaningless.
        _class_counts = sev_tr["failure_level"].value_counts() if len(sev_tr) > 0 else pd.Series([])
        _skip_stage2 = len(sev_tr) == 0 or _class_counts.min() < 20
        if _skip_stage2:
            print(f"  [{CATEGORY}] Stage 2 skipped — min class count "
                  f"{_class_counts.min() if len(_class_counts) else 0} < 20 (GATE level 2/3/4 sparse)")
        if not _skip_stage2 and len(sev_te) > 0:
            Xs_tr = sev_tr[FEATURE_COLS].values
            ys_tr = sev_tr["failure_level"].values.astype(int) - 1
            Xs_te = sev_te[FEATURE_COLS].values
            ys_te = sev_te["failure_level"].values.astype(int) - 1
            with mlflow.start_run(run_name=f"stage2_severity_{tag}") as run:
                mlflow.set_tags({**COMMON_TAGS,"model_type":"xgboost_multiclass","stage":"2","num_classes":"3"})
                sev_model = XGBClassifier(n_estimators=300,max_depth=5,learning_rate=0.05,
                                          objective="multi:softprob",num_class=3,
                                          tree_method="hist",random_state=42,n_jobs=-1)
                sev_model.fit(Xs_tr, ys_tr, verbose=False)
                sev_auc = roc_auc_score(ys_te, sev_model.predict_proba(Xs_te), multi_class="ovr", average="macro")
                sev_f1  = f1_score(ys_te, sev_model.predict(Xs_te), average="macro")
                mlflow.log_metrics({"test_severity_auc":round(sev_auc,4),"test_severity_f1":round(sev_f1,4)})
            print(f"  Stage2 Severity ({CATEGORY}) AUC={sev_auc:.4f}  F1={sev_f1:.4f}")

    # ── Fix 5: Threshold tuning on val PR curve ──────────────────────────────
    # PS1 prioritises recall — a missed failure costs more than a false alarm.
    # Tune threshold on VALIDATION PR curve targeting recall >= 0.80,
    # then pick the highest-precision threshold that meets that floor.
    y_prob_val_thresh = xgb_tuned.predict_proba(X_val)[:, 1]
    prec_curve, rec_curve, thresh_curve = precision_recall_curve(y_val, y_prob_val_thresh)
    RECALL_FLOOR = cat["recall_floor"]
    candidates = [(t, p, r) for p, r, t in zip(prec_curve[:-1], rec_curve[:-1], thresh_curve)
                  if r >= RECALL_FLOOR]
    if candidates:
        BEST_THRESHOLD, BEST_PREC, BEST_REC = max(candidates, key=lambda x: x[1])
    else:
        BEST_THRESHOLD, BEST_PREC, BEST_REC = 0.5, float("nan"), float("nan")
        print(f"  [{CATEGORY}] No threshold achieves recall>={RECALL_FLOOR} — using 0.5")
    print(f"  Threshold ({CATEGORY}): {BEST_THRESHOLD:.3f}  "
          f"prec={BEST_PREC:.3f}  rec={BEST_REC:.3f}  (recall>={RECALL_FLOOR} on val)")

    # ── Fix 6: Champion selection by test_ap (Average Precision) ─────────────
    # Under heavy imbalance (3-15% positive) ROC-AUC over-estimates skill.
    # Average Precision = area under the PR curve — far more discriminating here.
    all_runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME],
                                   filter_string="tags.stage != '2'",
                                   order_by=["metrics.test_ap DESC"])
    CHAMPION_RUN_ID = all_runs.iloc[0]["run_id"]
    CHAMPION_NAME   = all_runs.iloc[0]["tags.mlflow.runName"]
    CHAMPION_AUC    = all_runs.iloc[0]["metrics.test_auc"]
    CHAMPION_AP     = all_runs.iloc[0]["metrics.test_ap"]
    print(f"  * Champion ({CATEGORY}): {CHAMPION_NAME}  AUC={CHAMPION_AUC:.4f}  AP={CHAMPION_AP:.4f}")

    # ── MLflow Model Registry ─────────────────────────────────────────────────
    mlfc = mlflow.tracking.MlflowClient()
    try:
        mlfc.create_registered_model(name=MLFLOW_MODEL_NAME,
                                      description=f"PS1-3d {CATEGORY} Device Failure — Chicago Ventra. Binary: will_fail_3d. v6.",
                                      tags={"ps":"PS1_3d","category":CATEGORY,"target":"will_fail_3d","city":"chicago"})
    except mlflow.exceptions.MlflowException: pass
    mv = mlfc.create_model_version(name=MLFLOW_MODEL_NAME, source=f"runs:/{CHAMPION_RUN_ID}/model",
                                    run_id=CHAMPION_RUN_ID,
                                    description=f"Champion: {CHAMPION_NAME}  AUC={CHAMPION_AUC:.4f}  Train: 2024-01-01 to 2025-09-30",
                                    tags={"algorithm":CHAMPION_NAME,"test_auc":str(CHAMPION_AUC),"n_features":str(len(FEATURE_COLS))})
    mlfc.transition_model_version_stage(name=MLFLOW_MODEL_NAME, version=mv.version, stage="Staging")
    MLFLOW_VERSION = mv.version
    print(f"  MLflow {MLFLOW_MODEL_NAME} v{MLFLOW_VERSION} -> Staging")

    # ── SageMaker Model Registry ──────────────────────────────────────────────
    champion_model = mlflow.sklearn.load_model(f"runs:/{CHAMPION_RUN_ID}/model")
    joblib.dump(champion_model, f"/tmp/ps1_{tag}_champion.joblib")
    joblib.dump(FEATURE_COLS,   f"/tmp/ps1_{tag}_feature_cols.joblib")
    test_metrics_dict = evaluate(champion_model, X_test, y_test, "test")
    S3_CLIENT.put_object(Bucket=ARTIFACT_BUCKET,
                          Key=f"sagemaker/ps1-3d/{tag}/metrics/test_metrics.json",
                          Body=json.dumps(test_metrics_dict), ContentType="application/json")
    with tarfile.open(f"/tmp/model_{tag}.tar.gz", "w:gz") as tar:
        tar.add(f"/tmp/ps1_{tag}_champion.joblib",    arcname=f"ps1_{tag}_champion.joblib")
        tar.add(f"/tmp/ps1_{tag}_feature_cols.joblib",arcname=f"ps1_{tag}_feature_cols.joblib")
    MODEL_S3_KEY = f"sagemaker/ps1-3d/{tag}/model-v{MLFLOW_VERSION}/model.tar.gz"
    S3_CLIENT.upload_file(f"/tmp/model_{tag}.tar.gz", ARTIFACT_BUCKET, MODEL_S3_KEY)
    MODEL_DATA_URL = f"s3://{ARTIFACT_BUCKET}/{MODEL_S3_KEY}"

    import sagemaker.image_uris as sm_images
    image_uri = sm_images.retrieve(framework="sklearn", region=REGION, version="1.2-1",
                                    instance_type="ml.m5.large", image_scope="inference")
    try:
        SM_CLIENT.create_model_package_group(ModelPackageGroupName=MODEL_REGISTRY_NAME,
                                              ModelPackageGroupDescription=f"PS1-3d {CATEGORY} — Chicago Ventra. will_fail_3d. v6.")
    except Exception: pass
    resp = SM_CLIENT.create_model_package(
        ModelPackageGroupName=MODEL_REGISTRY_NAME,
        ModelPackageDescription=f"{CHAMPION_NAME} | AUC={CHAMPION_AUC:.4f} | MLflow v{MLFLOW_VERSION}",
        InferenceSpecification={
            "Containers":[{"Image":image_uri,"ModelDataUrl":MODEL_DATA_URL}],
            "SupportedContentTypes":["text/csv","application/json"],
            "SupportedResponseMIMETypes":["application/json"],
            "SupportedTransformInstanceTypes":["ml.m5.large"],
            "SupportedRealtimeInferenceInstanceTypes":["ml.m5.large"],
        },
        ModelApprovalStatus="PendingManualApproval",
        ModelMetrics={"ModelQuality":{"Statistics":{"ContentType":"application/json",
                      "S3Uri":f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/metrics/test_metrics.json"}}},
        CustomerMetadataProperties={
            "ps":"PS1_3d","category":CATEGORY,"target":"will_fail_3d","city":"chicago",
            "algorithm":CHAMPION_NAME,"test_auc":str(round(CHAMPION_AUC,4)),
            "mlflow_version":str(MLFLOW_VERSION),"mlflow_run_id":CHAMPION_RUN_ID,
            "n_features":str(len(FEATURE_COLS))})
    SM_MODEL_PKG_ARN = resp["ModelPackageArn"]
    print(f"  SM Package ({CATEGORY}): ...{SM_MODEL_PKG_ARN[-40:]}")

    CATEGORY_RESULTS[CATEGORY] = {
        "champion_run_id": CHAMPION_RUN_ID, "champion_name": CHAMPION_NAME,
        "champion_auc": CHAMPION_AUC, "champion_ap": CHAMPION_AP,
        "threshold": BEST_THRESHOLD, "threshold_prec": BEST_PREC, "threshold_rec": BEST_REC,
        "mlflow_version": MLFLOW_VERSION,
        "sm_pkg_arn": SM_MODEL_PKG_ARN, "model_data_url": MODEL_DATA_URL,
        "champion_model": champion_model,
        "X_test": X_test, "y_test": y_test, "X_train": X_train,
    }

print(f"\n{'='*65}")
print("  Training complete — TVM and GATE done")
for cn, res in CATEGORY_RESULTS.items():
    print(f"  {cn:<6}  champion={res['champion_name']:<30}  "
          f"AUC={res['champion_auc']:.4f}  AP={res['champion_ap']:.4f}  "
          f"thresh={res['threshold']:.3f}")
print(f"{'='*65}")


In [ ]:
# ── CELL 11 : Approve + deploy SageMaker Endpoints (per category) ────────────
from sagemaker.model import ModelPackage
from sagemaker.model_monitor import DataCaptureConfig

PREDICTORS = {}
for CATEGORY, res in CATEGORY_RESULTS.items():
    cat = CATEGORIES[CATEGORY]
    SM_CLIENT.update_model_package(ModelPackageArn=res["sm_pkg_arn"], ModelApprovalStatus="Approved")
    print(f"[{CATEGORY}] Approved.")
    registered = ModelPackage(role=ROLE, model_package_arn=res["sm_pkg_arn"], sagemaker_session=SM_SESSION)
    predictor = registered.deploy(
        initial_instance_count=1, instance_type="ml.m5.large",
        endpoint_name=cat["endpoint"],
        data_capture_config=DataCaptureConfig(
            enable_capture=True, sampling_percentage=100,
            destination_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{CATEGORY.lower()}/data-capture/",
            capture_options=["Input","Output"]),
    )
    PREDICTORS[CATEGORY] = predictor
    print(f"[{CATEGORY}] Endpoint live: {cat['endpoint']}")
    sample_csv = pd.DataFrame(res["X_test"][:3], columns=FEATURE_COLS).to_csv(index=False, header=False)
    result = predictor.predict(sample_csv, initial_args={"ContentType":"text/csv"})
    print(f"[{CATEGORY}] Sample predictions: {result}")


In [ ]:
# ── CELL 12 : SageMaker Model Monitor — Data Quality Baselines (per category) ─
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat
from sagemaker.model_monitor import CronExpressionGenerator

MONITORS = {}
for CATEGORY, res in CATEGORY_RESULTS.items():
    tag = CATEGORY.lower()
    monitor = DefaultModelMonitor(role=ROLE, instance_count=1, instance_type="ml.m5.large",
                                   volume_size_in_gb=20, max_runtime_in_seconds=3600,
                                   sagemaker_session=SM_SESSION)
    baseline_df = pd.DataFrame(res["X_train"], columns=FEATURE_COLS)
    baseline_df.to_csv(f"/tmp/baseline_{tag}.csv", index=False)
    S3_CLIENT.upload_file(f"/tmp/baseline_{tag}.csv", ARTIFACT_BUCKET,
                          f"sagemaker/ps1-3d/{tag}/baseline/baseline.csv")
    monitor.suggest_baseline(
        baseline_dataset=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/baseline/baseline.csv",
        dataset_format=DatasetFormat.csv(header=True),
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/baseline/output/",
        wait=True, logs=True)
    print(f"[{CATEGORY}] Baseline complete.")
    MONITORS[CATEGORY] = monitor
print("All baselines done.")


In [ ]:
# ── CELL 13 : Schedule daily Data Quality monitoring (per category) ───────────
cw = boto3.client("cloudwatch", region_name=REGION)

for CATEGORY, monitor in MONITORS.items():
    tag = CATEGORY.lower()
    cat = CATEGORIES[CATEGORY]
    sched_name = f"chicago-ps1-3d-{tag}-data-quality"
    monitor.create_monitoring_schedule(
        monitor_schedule_name=sched_name, endpoint_input=cat["endpoint"],
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/monitor-output/",
        statistics=monitor.baseline_statistics(), constraints=monitor.suggested_constraints(),
        schedule_cron_expression=CronExpressionGenerator.daily(), enable_cloudwatch_metrics=True)
    print(f"[{CATEGORY}] Data Quality Monitor: {sched_name}")
    cw.put_metric_alarm(
        AlarmName=f"PS1-3d-{CATEGORY}-DataQuality-ViolationRate",
        ComparisonOperator="GreaterThanThreshold", EvaluationPeriods=1,
        MetricName="feature_baseline_drift_percentage",
        Namespace="aws/sagemaker/Endpoints/data-metrics",
        Period=86400, Statistic="Average", Threshold=3.0,
        AlarmDescription=f"PS1-3d {CATEGORY} input feature violation rate > 3%",
        Dimensions=[{"Name":"Endpoint","Value":cat["endpoint"]}],
        TreatMissingData="notBreaching")
    print(f"[{CATEGORY}] CW Alarm: PS1-3d-{CATEGORY}-DataQuality-ViolationRate")


In [ ]:
# ── CELL 14 : Model Quality Monitor (per category) ──────────────────────────
from sagemaker.model_monitor import ModelQualityMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

MQ_MONITORS = {}
for CATEGORY, res in CATEGORY_RESULTS.items():
    tag = CATEGORY.lower()
    cat = CATEGORIES[CATEGORY]
    mq = ModelQualityMonitor(role=ROLE, instance_count=1, instance_type="ml.m5.large",
                              sagemaker_session=SM_SESSION)
    y_prob_test = res["champion_model"].predict_proba(res["X_test"])[:,1]
    mq_df = pd.DataFrame({"prediction":y_prob_test, "probability":y_prob_test,
                           "label":res["y_test"].astype(str)})
    mq_df.to_csv(f"/tmp/mq_baseline_{tag}.csv", index=False)
    S3_CLIENT.upload_file(f"/tmp/mq_baseline_{tag}.csv", ARTIFACT_BUCKET,
                          f"sagemaker/ps1-3d/{tag}/mq-baseline/mq_baseline.csv")
    mq.suggest_baseline(
        baseline_dataset=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/mq-baseline/mq_baseline.csv",
        dataset_format=DatasetFormat.csv(header=True),
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/mq-baseline/output/",
        problem_type="BinaryClassification",
        inference_attribute="prediction", probability_attribute="probability",
        ground_truth_attribute="label", probability_threshold_attribute="0.5", wait=True)
    mq.create_monitoring_schedule(
        monitor_schedule_name=f"chicago-ps1-3d-{tag}-model-quality",
        endpoint_input=cat["endpoint"],
        ground_truth_input=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/ground-truth/",
        problem_type="BinaryClassification",
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/mq-output/",
        constraints=mq.suggested_constraints(),
        schedule_cron_expression=CronExpressionGenerator.daily(), enable_cloudwatch_metrics=True)
    print(f"[{CATEGORY}] Model Quality Monitor: chicago-ps1-3d-{tag}-model-quality")
    print(f"  Ground truth: s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/ground-truth/")
    MQ_MONITORS[CATEGORY] = mq


In [ ]:
# ── CELL 15 : Final summary (per category) ────────────────────────────────────
print("="*65)
print("PS1-3d TVM+GATE -- Complete  (CUBIC MARS AWS v6)")
print("="*65)

for CATEGORY, res in CATEGORY_RESULTS.items():
    cat = CATEGORIES[CATEGORY]
    tag = CATEGORY.lower()
    print(f"\n-- {CATEGORY} --")
    print(f"  Champion       : {res['champion_name']}  AUC={res['champion_auc']:.4f}")
    print(f"  MLflow model   : {cat['model_name']}  v{res['mlflow_version']}  [Staging]")
    print(f"  SM Registry    : {cat['registry_name']}")
    print(f"  SM Endpoint    : {cat['endpoint']}")
    print(f"  Feature Store  : {cat['fs_group']}")
    print(f"  Data Monitor   : chicago-ps1-3d-{tag}-data-quality  (daily)")
    print(f"  Model Monitor  : chicago-ps1-3d-{tag}-model-quality  (daily)")
    print(f"  CW Alarm       : PS1-3d-{CATEGORY}-DataQuality-ViolationRate")

print(f"\n  Features       : {len(FEATURE_COLS)}")
print(f"  Artifacts      : s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/")
print("""
Retraining flow:
  1. Re-run from CELL 4 -- reads latest Gold, trains both TVM + GATE
  2. MLflow auto-increments version numbers per category
  3. SM Registry adds new packages (PendingManualApproval)
  4. Approve in Studio UI -> new Endpoints deployed
""")
